In [15]:
"""
  ______    ____     ____    _______   ______   _____          _____               _______   ______ 
 |  ____|  / __ \   / __ \  |__   __| |  ____| |  __ \        |  __ \      /\     |__   __| |  ____|
 | |__    | |  | | | |  | |    | |    | |__    | |__) |       | |  | |    /  \       | |    | |__   
 |  __|   | |  | | | |  | |    | |    |  __|   |  _  /        | |  | |   / /\ \      | |    |  __|  
 | |      | |__| | | |__| |    | |    | |____  | | \ \        | |__| |  / ____ \     | |    | |____ 
 |_|       \____/   \____/     |_|    |______| |_|  \_\       |_____/  /_/    \_\    |_|    |______|
"""





import os
import re
from datetime import datetime
from pathlib import Path

def mettre_a_jour_footers():
    # 1. Obtenir la date et l'heure au format : JJ/MM/AAAA à HHhMM (ex: 29/10/2023 à 08h05)
    maintenant = datetime.now()
    date_heure_formatee = maintenant.strftime("%d/%m/%Y à %Hh%M")
    
    nouveau_footer = f'<footer class="main-footer">\n        <p>Mis à jour le {date_heure_formatee}</p>\n    </footer>'
    
    # 2. Cibler la racine et le dossier pages
    fichiers_html = []
    
    # Ajoute l'index.html à la racine s'il existe
    if os.path.exists("index.html"):
        fichiers_html.append(Path("index.html"))
        
    # Ajoute tous les fichiers .html du dossier pages/
    dossier_pages = Path("pages")
    if dossier_pages.exists():
        fichiers_html.extend(dossier_pages.glob("*.html"))
        
    compteur = 0
    
    # 3. Pattern pour détecter le bloc <footer class="main-footer">...</footer>
    pattern_footer = re.compile(r'<footer class="main-footer">.*?</footer>', re.DOTALL)
    
    for chemin in fichiers_html:
        with open(chemin, "r", encoding="utf-8") as f:
            contenu = f.read()
            
        # On vérifie si le footer existe bien dans la page avant de remplacer
        if pattern_footer.search(contenu):
            contenu_modifie = pattern_footer.sub(nouveau_footer, contenu)
            
            with open(chemin, "w", encoding="utf-8") as f:
                f.write(contenu_modifie)
            #print(f"✅ Footer mis à jour dans : {chemin}")
            compteur += 1
        else:
            pass
            #print(f"⚠️ Aucun footer trouvé dans : {chemin}")
            
    print(f"\nTerminé ! {compteur} page(s) mise(s) à jour avec le message : Mis à jour le {date_heure_formatee}")

In [16]:
""" 
 __      __   ____     _____              ____  
 \ \    / /  / __ \   / ____|     /\     |  _ \ 
  \ \  / /  | |  | | | |         /  \    | |_) |
   \ \/ /   | |  | | | |        / /\ \   |  _ < 
    \  /    | |__| | | |____   / ____ \  | |_) |
     \/      \____/   \_____| /_/    \_\ |____/ 
"""





import os
import re
from pathlib import Path

# --- Configuration des chemins ---
ROOT_DIR = Path.cwd().absolute()
FILES_DIR = ROOT_DIR / "files" / "Anglais"
PAGES_DIR = ROOT_DIR / "pages"
TEMPLATE_PATH = ROOT_DIR / "template.html"

# Cible en_vocab.html directement dans le dossier pages/
EN_VOCAB_ROOT_PATH = PAGES_DIR / "en_vocab.html"

# Créer le dossier pages s'il n'existe pas
PAGES_DIR.mkdir(parents=True, exist_ok=True)

# Charger le modèle HTML
with open(TEMPLATE_PATH, "r", encoding="utf-8") as f:
    TEMPLATE_CONTENT = f.read()

def formater_nom_fichier(chaine):
    """Nettoie une chaîne pour en faire un nom de fichier web propre (slug)"""
    chaine = chaine.lower()
    chaine = re.sub(r'[^\w\s-]', '', chaine)
    return re.sub(r'[-\s]+', '-', chaine).strip('-')

def construire_chemin_page(segments, est_fiche=False, est_racine_vocab=False):
    """Génère le chemin complet du fichier HTML dans PAGES_DIR sans la partie française"""
    if est_racine_vocab:
        return EN_VOCAB_ROOT_PATH
        
    theme_slugs = [formater_nom_fichier(s) for s in segments[:-1]] if est_fiche else [formater_nom_fichier(s) for s in segments]
    
    nom_base = "en_vocab_" + "_".join(theme_slugs)
    if est_fiche:
        nom_fiche_brut = segments[-1]
        # RETRAIT DU FRANÇAIS : Si un ';' est présent, on extrait uniquement la partie anglaise
        if ';' in nom_fiche_brut:
            nom_fiche_brut = nom_fiche_brut.split(';', 1)[0].strip()
            
        nom_base += f"_fiche-{formater_nom_fichier(nom_fiche_brut)}"
        
    return PAGES_DIR / f"{nom_base}.html"

def convertir_type(v_type):
    """Transforme les codes de types en libellés propres"""
    v_type = v_type.strip().upper()
    mapping = {
        'N': 'noun',
        'V': 'verb',
        'VI': 'irr. verb',
        'ADJ': 'adj.',
        'ABREV': 'abrev.',
        'A': 'other',
        'DEF': 'notion'
    }
    return mapping.get(v_type, v_type.lower())

def convertir_provenance(prov):
    """Transforme les codes de provenance en libellés propres"""
    prov = prov.strip().upper()
    mapping = {
        'LIVRE': 'Livre',
        'LIVREM': 'Marge du livre',
        'COURS': 'Vu en cours',
        'SUPPTHEME': 'Supplément du thème',
        'SUPPLIVRE': 'Livre (pas vu)',
        'SUPP': 'Supplément'
    }
    return mapping.get(prov, prov)

def generer_breadcrumbs(segments, est_fiche=False, est_racine_vocab=False):
    """Construit le fil d'Ariane adapté à une exécution où toutes les pages sont dans /pages"""
    breadcrumbs = ['<a href="../index.html">Accueil</a> > <a href="./en.html">Anglais</a>']
    
    if est_racine_vocab:
        breadcrumbs.append('<a href="./en_vocab.html">Vocabulaire</a>')
        return " > ".join(breadcrumbs)
        
    breadcrumbs.append('<a href="./en_vocab.html">Vocabulaire</a>')
    
    for i in range(len(segments)):
        current_segments = segments[:i+1]
        is_last = (i == len(segments) - 1)
        
        if is_last and est_fiche:
            target_file = construire_chemin_page(segments, est_fiche=True).name
        else:
            target_file = construire_chemin_page(current_segments, est_fiche=False).name
            
        segment_brut = segments[i]
        
        # --- MODIFICATION ICI : On ne garde que la partie Anglaise si un ';' est présent ---
        if ';' in segment_brut:
            segment_brut = segment_brut.split(';', 1)[0]
            
        titre_segment = segment_brut.replace("-", " ").strip().capitalize()
        breadcrumbs.append(f'<a href="./{target_file}\">{titre_segment}</a>')
        
    return " > ".join(breadcrumbs)


def formater_lien_exo(lien_exo):
    lien_exo_formate = lien_exo.split("fiche-")
    lien_exo_formate = lien_exo_formate[1][:-5].capitalize()
    return lien_exo_formate.replace("-", " ")

        

# --- 1. Cartographie de l'arborescence ---
arborescence = {}
themes_principaux = set()

for root, dirs, files in os.walk(FILES_DIR):
    path_root = Path(root)
    relative_parts = path_root.relative_to(FILES_DIR).parts
    
    if not relative_parts:
        for d in dirs:
            themes_principaux.add(d)
            if (d,) not in arborescence:
                arborescence[(d,)] = {'sous_themes': set(), 'fiches': set()}
        continue

    current_node = relative_parts
    if current_node not in arborescence:
        arborescence[current_node] = {'sous_themes': set(), 'fiches': set()}

    for d in dirs:
        arborescence[current_node]['sous_themes'].add(d)
        child_node = current_node + (d,)
        if child_node not in arborescence:
            arborescence[child_node] = {'sous_themes': set(), 'fiches': set()}
            
    for f in files:
        if f.endswith('.txt'):
            arborescence[current_node]['fiches'].add(f[:-4])

# --- 2. Génération des pages ---

# -- A. Génération de la page racine pages/en_vocab.html --
corps_racine = '<p class="intro-text">Les fiches sont triées par thèmes, elles font en général une quarantaine de mots pour que ça fasse pas trop lourd à apprendre</p>\n'
corps_racine += """
    <a href="./en_tree.html" class="btn-tree-link" style="text-decoration: none;">
        <button class="btn-tree-toggle">🌳 Arborescence des fiches</button>
    </a>
    """
corps_racine += '<section class="subject-section">\n<h3>Thèmes principaux</h3>\n<div class="topic-grid">\n'
for t in sorted(themes_principaux):
    lien_cible = f"./{construire_chemin_page((t,), est_fiche=False).name}"
    nom_image = f"en_vocab_{formater_nom_fichier(t)}.png"
    
    

    corps_racine += f"""
    <a href="{lien_cible}" class="subject-topic">
        <span class="topic-title">{t.replace("-", " ").capitalize()}</span>
        <img src="../images/{nom_image}" alt="img {t}" class="topic-image">
    </a>\n"""
corps_racine += '</div>\n</section>\n'

html_racine = TEMPLATE_CONTENT.format(
    PAGE_TITLE="Vocabulaire",
    MAIN_TITLE="ANGLAIS > Vocabulaire",
    BREADCRUMBS=generer_breadcrumbs([], est_racine_vocab=True),
    CONTENT=corps_racine
)
with open(EN_VOCAB_ROOT_PATH, "w", encoding="utf-8") as f_out:
    f_out.write(html_racine)


# -- B. Génération des Fiches de Vocabulaire --
for node, data in arborescence.items():
    for fiche in data['fiches']:
        chemin_txt = FILES_DIR / os.path.join(*node) / f"{fiche}.txt"
        segments_fiche = list(node) + [fiche]
        fichier_html_cible = construire_chemin_page(segments_fiche, est_fiche=True)
        
        if ';' in fiche:
            partie_en, partie_fr = fiche.split(';', 1)
            titre_en = partie_en.replace("-", " ").strip().capitalize()
            titre_fr = partie_fr.replace("-", " ").strip().capitalize()
            titre_propre = f"{titre_en} - {titre_fr}"
        else:
            titre_propre = fiche.replace("-", " ").strip().capitalize()
        
        tableau_html = '<table class="vocab-table">\n<thead>\n<tr>'
        tableau_html += '<th>Type</th><th>Anglais</th><th>Français</th><th>Niveau</th><th>Note</th><th>Exemple</th><th>Provenance</th>'
        tableau_html += '</tr>\n</thead>\n<tbody>\n'
        
        lien_quizlet = None
        liens_exos = []
        compteur_mots = 0
        
        with open(chemin_txt, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                if line.startswith("QUIZLET;"):
                    parts_lien = line.split(';')
                    lien_quizlet = parts_lien[1].strip() if len(parts_lien) > 1 else parts_lien[0].strip()
                    continue
                
                if line.startswith("EXOS;"):
                    parts_lien = line.split(';')
                    liens_exos.append(parts_lien[1].strip() if len(parts_lien) > 1 else parts_lien[0].strip())
                    continue
                
                if line.startswith("#"):
                    continue
                
                parts = [p.strip() for p in line.split(';')]
                if len(parts) < 7:
                    continue
                
                compteur_mots += 1
                
                anglais = parts[0]
                francais = parts[1]
                note = parts[2]
                exemple = parts[3]
                v_type = parts[4]
                provenance = parts[5]
                niveau = parts[6]
                
                type_propre = convertir_type(v_type)
                provenance_propre = convertir_provenance(provenance)
                
                note_propre = "" if note == "..." else note
                exemple_propre = "" if exemple == "..." else exemple
                
                type_classe = formater_nom_fichier(type_propre)
                niveau_classe = niveau.strip().lower()

                cell_type = f"<span class='badge badge-type badge-type-{type_classe}'>{type_propre}</span>" if type_propre != "..." else ""
                cell_anglais = f"<span class='txt-anglais'>{anglais}</span>"
                cell_francais = f"<span class='txt-francais'>{francais}</span>"
                cell_level = f"<span class='badge badge-level badge-level-{niveau_classe}'>{niveau}</span>" if niveau != "..." else ""
                cell_note = f"<span class='txt-note'>{note_propre}</span>"
                cell_exemple = f"<span class='txt-exemple'>{exemple_propre}</span>"
                cell_prov = f"<span class='badge badge-prov'>{provenance_propre}</span>" if provenance_propre != "..." else ""
                
                tableau_html += f"<tr><td>{cell_type}</td><td>{cell_anglais}</td><td>{cell_francais}</td><td>{cell_level}</td><td>{cell_note}</td><td>{cell_exemple}</td><td>{cell_prov}</td></tr>\n"
        
        tableau_html += "</tbody>\n</table>"
        
        titre_avec_compteur = f"{titre_propre} / {compteur_mots} termes"
        
        # --- 1. Bouton Quizlet conditionnel ---
        html_bouton_quizlet = ""
        if lien_quizlet:
            html_bouton_quizlet = f"""
            <a href="{lien_quizlet}" target="_blank" class="action-button-link quizlet-btn" title="Ouvrir dans Quizlet">
                <img src="../images/quizlet_logo.png" alt="Quizlet">
            </a>
            """
        
        # --- CALCUL DU NOM DE FICHIER UNIQUE (SANS LE FRANÇAIS) ---
        # On utilise le titre anglais nettoyé pour les fichiers physiques .txt
        nom_fiche_en_uniquement = fiche.split(';')[0].strip()
        theme_slugs = [formater_nom_fichier(s) for s in node]
        suffixe_nom_txt = f"en_vocab_{'_'.join(theme_slugs)}_liste-{formater_nom_fichier(nom_fiche_en_uniquement)}.txt"
        
        # --- 2. Bouton Liste Brute (Dossier renommé en texts-quizlet-lists) ---
        chemin_relatif_quizlet_list = f"../texts-quizlet-lists/{suffixe_nom_txt}"
        html_bouton_source = f"""
        <a href="{chemin_relatif_quizlet_list}" target="_blank" class="action-button-link" download title="Télécharger la liste brute (.txt)">
            <img src="../images/txt_quizlet_icon.png" alt="TXT Source">
        </a>
        """
        
        # --- 3. Bouton Fichier Import Anki ---
        chemin_relatif_anki_list = f"../texts-anki-lists/{suffixe_nom_txt}"
        html_bouton_anki = f"""
        <a href="{chemin_relatif_anki_list}" target="_blank" class="action-button-link anki-btn" download title="Télécharger le fichier d'import Anki (.txt)">
            <img src="../images/txt_anki_icon.png" alt="Anki Source">
        </a>
        """
        
        # --- 4. Assemblage de l'en-tête avec la ligne d'actions ---
        corps_page = f"""
        <div class="fiche-header-container">
            <div class="fiche-titles">
                <p class="intro-text">Fiche de vocabulaire</p>
                <h3 class="fiche-title">{titre_avec_compteur}</h3>
            </div>
            
            <div class="fiche-actions-inline">
                {html_bouton_quizlet}
                {html_bouton_source}
                {html_bouton_anki}
            </div>
        </div>
        
        {tableau_html}

        <hr class="section-divider">
        <div class="exercises-section">
            <h3>Exercices en liens avec cette fiche :</h3>
            <div class="liens-exo-inline">
        """
        
        for lien_exo in liens_exos:
            corps_page += f"""
            <a href="{lien_exo}" class="action-button-link exo-btn" title="Exercice">{formater_lien_exo(lien_exo)}</a>
            """

        corps_page += """
            </div>
        </div>
        """


        breadcrumbs_html = generer_breadcrumbs(segments_fiche, est_fiche=True)
        html_final = TEMPLATE_CONTENT.format(
            PAGE_TITLE=titre_avec_compteur,
            MAIN_TITLE="ANGLAIS > Vocabulaire",
            BREADCRUMBS=breadcrumbs_html,
            CONTENT=corps_page
        )
        
        with open(fichier_html_cible, "w", encoding="utf-8") as f_out:
            f_out.write(html_final)

# -- C. Génération des pages de Navigation --
for node, data in arborescence.items():
    fichier_html_cible = construire_chemin_page(node, est_fiche=False)
    titre_navigation = node[-1].replace("-", " ").capitalize()
    
    corps_page = ""
    
    if data['sous_themes']:
        corps_page += '<section class="subject-section">\n<h3>Sous-thèmes disponibles</h3>\n<div class="topic-grid">\n'
        for st in sorted(data['sous_themes']):
            node_cible = node + (st,)
            lien_cible = f"./{construire_chemin_page(node_cible, est_fiche=False).name}"
            theme_slugs = [formater_nom_fichier(s) for s in node_cible]
            nom_image = f"en_vocab_{'_'.join(theme_slugs)}.png"
            corps_page += f"""<a href="{lien_cible}" class="subject-topic"><span class="topic-title">{st.replace("-", " ").capitalize()}</span><img src="../images/{nom_image}" alt="img {st}" class="topic-image"></a>\n"""
        corps_page += '</div>\n</section>\n'
        
    if data['fiches']:
        corps_page += '<section class="fiches-section">\n<h3>Listes de vocabulaire</h3>\n<ul class="subject-list">\n'
        for fiche in sorted(data['fiches']):
            segments_fiche = list(node) + [fiche]
            lien_cible = f"./{construire_chemin_page(segments_fiche, est_fiche=True).name}"
            titre_fiche = fiche.split(';')[0].replace("-", " ").capitalize() if ';' in fiche else fiche.replace("-", " ").capitalize()
            corps_page += f'<li><a href="{lien_cible}" class="subject-link">📄 {titre_fiche}</a></li>\n'
        corps_page += '</ul>\n</section>\n'
        
    breadcrumbs_html = generer_breadcrumbs(node, est_fiche=False)
    html_final = TEMPLATE_CONTENT.format(
        PAGE_TITLE=titre_navigation,
        MAIN_TITLE="ANGLAIS > Vocabulaire",
        BREADCRUMBS=breadcrumbs_html,
        CONTENT=corps_page
    )
    
    with open(fichier_html_cible, "w", encoding="utf-8") as f_out:
        f_out.write(html_final)


mettre_a_jour_footers()
print("Génération réussie avec support des liens Quizlet !")


Terminé ! 159 page(s) mise(s) à jour avec le message : Mis à jour le 08/07/2026 à 18h39
Génération réussie avec support des liens Quizlet !


In [17]:
"""
             _____    ____     ____    _____    ______    _____    _____   ______   _   _    _____   ______ 
     /\     |  __ \  |  _ \   / __ \  |  __ \  |  ____|  / ____|  / ____| |  ____| | \ | |  / ____| |  ____|
    /  \    | |__) | | |_) | | |  | | | |__) | | |__    | (___   | |      | |__    |  \| | | |      | |__   
   / /\ \   |  _  /  |  _ <  | |  | | |  _  /  |  __|    \___ \  | |      |  __|   | . ` | | |      |  __|  
  / ____ \  | | \ \  | |_) | | |__| | | | \ \  | |____   ____) | | |____  | |____  | |\  | | |____  | |____ 
 /_/    \_\ |_|  \_\ |____/   \____/  |_|  \_\ |______| |_____/   \_____| |______| |_| \_|  \_____| |______|
"""





import os
import re
from pathlib import Path

def generer_page_arbre_statique(base_dir="files/Anglais/", output_file="pages/en_tree.html"):
    path_anglais = Path(base_dir)
    if not path_anglais.exists():
        print(f"❌ Dossier {base_dir} introuvable.")
        return

    # Nettoyage standard : remplace tous les espaces par des tirets du 6 (-)
    def clean_slug_dash(text):
        t = text.lower().strip()
        t = re.sub(r'[^a-z0-9\s-]', '', t)
        t = re.sub(r'[\s_]+', '-', t)
        return t

    # Fonction récursive pour explorer les dossiers
    def explorer_dossier_rec(dossier_actuel, slug_parent=""):
        lignes_html = []
        
        # Récupération et tri des éléments
        items = sorted(list(dossier_actuel.iterdir()))
        if not items:
            return lignes_html

        lignes_html.append('<ul>')
        for item in items:
            if item.is_dir():
                nom_dossier = item.name
                slug_dossier = clean_slug_dash(nom_dossier)
                
                # Construction du chemin de slugs accumulés avec underscore (_) entre les dossiers
                nouveau_slug_chemin = f"{slug_parent}_{slug_dossier}" if slug_parent else slug_dossier
                
                # Lien vers la page du thème parent principal
                lien_theme_racine = f"./en_vocab_{slug_parent.split('_')[0] if slug_parent else slug_dossier}.html"
                
                # Structure cruciale : Un nœud (li) contient son bouton ET sa sous-liste (ul)
                lignes_html.append(f'    <li>')
                lignes_html.append(f'        <a href="{lien_theme_racine}" class="tree-link node-sub">📂 {nom_dossier}</a>')
                
                # APPEL RÉCURSIF pour descendre dans le sous-dossier
                lignes_html.extend(explorer_dossier_rec(item, nouveau_slug_chemin))
                
                lignes_html.append(f'    </li>')
                
            elif item.suffix == ".txt":
                partie_en = item.stem.split(';')[0].strip()
                fiche_slug = clean_slug_dash(partie_en)
                
                # FORMAT MIXTE : en_vocab_DOSSIER1_DOSSIER2_fiche-NOMFICHE.html
                if slug_parent:
                    lien_fiche = f"./en_vocab_{slug_parent}_fiche-{fiche_slug}.html"
                else:
                    lien_fiche = f"./en_vocab_root_fiche-{fiche_slug}.html"
                    
                lignes_html.append(f'    <li><a href="{lien_fiche}" class="tree-link">📄 {partie_en}</a></li>')
                
        lignes_html.append('</ul>')
        return lignes_html

    # --- DÉBUT DE LA GÉNÉRATION DE LA PAGE HTML ---
    html = []
    html.append("""<!DOCTYPE html>
<html lang="fr">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Carte du Vocabulaire - Prépa Blois</title>
    <link rel="stylesheet" href="../style.css">
</head>
<body>

    <header class="main-header">
        <h1><a href="../index.html">Prépa Blois</a></h1>
    </header>

    <main class="container">
        <h2 class="page-title">ANGLAIS > Carte du Vocabulaire</h2>
        
        <h4 class="breadcrumbs">
            <a href="../index.html">Accueil</a> >
            <a href="./en.html">Anglais</a> > 
            <a href="./en_vocab.html">Vocabulaire</a> > 
            <span>Carte (Arbre)</span>
        </h4>

        <p class="intro-text">Voici l'arborescence de toutes les fiches</p>
        
        <div class="tree-page-container" id="treePageContainer">
            <svg class="tree-svg-canvas" id="treeSvgCanvas"></svg>
            <ul class="tree-root">""")

    # Premier niveau (les thèmes racines)
    for racine_item in sorted(path_anglais.iterdir()):
        if racine_item.is_dir():
            nom_racine = racine_item.name
            slug_racine = clean_slug_dash(nom_racine)
            lien_theme_racine = f"./en_vocab_{slug_racine}.html"
            
            html.append(f'                <li>')
            html.append(f'                    <a href="{lien_theme_racine}" class="tree-link node-theme">📁 {nom_racine}</a>')
            
            # Lancement de la récursivité pour l'intérieur
            enfants_html = explorer_dossier_rec(racine_item, slug_racine)
            html.extend(enfants_html)
            
            html.append('                </li>')
            
        elif racine_item.suffix == ".txt":
            partie_en = racine_item.stem.split(';')[0].strip()
            fiche_slug = clean_slug_dash(partie_en)
            lien_fiche = f"./en_vocab_root_fiche-{fiche_slug}.html"
            html.append(f'                <li><a href="{lien_fiche}" class="tree-link">📄 {partie_en}</a></li>')

    # Fin de page et script JavaScript mis à jour
    html.append("""            </ul>
        </div>
    </main>

    <footer class="main-footer">
        <p>Mis à jour le --/--/---- à --h--</p>
    </footer>

    <script>
    function dessinerLignesArbre() {
        const svg = document.getElementById('treeSvgCanvas');
        const container = document.getElementById('treePageContainer');
        if (!svg || !container) return;
        
        svg.innerHTML = '';
        svg.setAttribute('width', '0');
        svg.setAttribute('height', '0');
        
        const totalWidth = Math.max(container.scrollWidth, container.clientWidth);
        const totalHeight = Math.max(container.scrollHeight, container.clientHeight);
        
        svg.setAttribute('width', totalWidth);
        svg.setAttribute('height', totalHeight);
        svg.style.width = totalWidth + 'px';
        svg.style.height = totalHeight + 'px';
        
        const containerRect = container.getBoundingClientRect();
        
        // On cible tous les LI qui contiennent directement un bouton ET une liste UL d'enfants
        const parents = container.querySelectorAll('li');
        
        parents.forEach(parentLi => {
            const boutonParent = parentLi.querySelector(':scope > .tree-link');
            const listeEnfants = parentLi.querySelector(':scope > ul');
            
            if (boutonParent && listeEnfants) {
                const enfantsLi = listeEnfants.querySelectorAll(':scope > li');
                
                enfantsLi.forEach(enfantLi => {
                    const boutonEnfant = enfantLi.querySelector(':scope > .tree-link');
                    
                    if (boutonEnfant) {
                        const pRect = boutonParent.getBoundingClientRect();
                        const cRect = boutonEnfant.getBoundingClientRect();
                        
                        const xDepart = (pRect.left + pRect.width) - containerRect.left;
                        const yDepart = (pRect.top + pRect.height / 2) - containerRect.top;
                        
                        const xArrivee = cRect.left - containerRect.left;
                        const yArrivee = (cRect.top + cRect.height / 2) - containerRect.top;
                        
                        const ligne = document.createElementNS('http://www.w3.org/2000/svg', 'path');
                        const inflechie = (xDepart + xArrivee) / 2;
                        
                        const d = `M ${xDepart} ${yDepart} C ${inflechie} ${yDepart}, ${inflechie} ${yArrivee}, ${xArrivee} ${yArrivee}`;
                        
                        ligne.setAttribute('d', d);
                        ligne.setAttribute('stroke', '#a0aec0'); 
                        ligne.setAttribute('stroke-width', '2');
                        ligne.setAttribute('fill', 'none');
                        
                        svg.appendChild(ligne);
                    }
                });
            }
        });
    }

    window.addEventListener('load', () => {
        setTimeout(dessinerLignesArbre, 350); // Léger délai accru pour assurer le positionnement CSS final
    });
    window.addEventListener('resize', dessinerLignesArbre);
    </script>

</body>
</html>""")

    chemin_destination = Path(output_file)
    chemin_destination.parent.mkdir(parents=True, exist_ok=True)
    with open(chemin_destination, "w", encoding="utf-8") as f:
        f.write("\n".join(html))
    print(f"✅ Page récursive stabilisée générée : {output_file}")

generer_page_arbre_statique()

✅ Page récursive stabilisée générée : pages/en_tree.html


In [18]:
"""
  ______  __   __   ____     _____ 
 |  ____| \ \ / /  / __ \   / ____|
 | |__     \ V /  | |  | | | (___  
 |  __|     > <   | |  | |  \___ \ 
 | |____   / . \  | |__| |  ____) |
 |______| /_/ \_\  \____/  |_____/ 
"""





import os
import re
from pathlib import Path

def generer_fiches_exercices(base_dir="files/Anglais Exo/", output_dir="pages/"):
    path_exos = Path(base_dir)
    dest_dir = Path(output_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    if not path_exos.exists():
        print(f"❌ Dossier {base_dir} introuvable.")
        return

    def clean_slug(text):
        t = text.lower().strip()
        t = re.sub(r'[^a-z0-9\s-]', '', t)
        t = re.sub(r'[\s_]+', '-', t)
        return t

    def explorer_et_generer(dossier_actuel, slug_parent=""):
        for item in dossier_actuel.iterdir():
            if item.is_dir():
                slug_dossier = clean_slug(item.name)
                nouveau_slug = f"{slug_parent}_{slug_dossier}" if slug_parent else slug_dossier
                explorer_et_generer(item, nouveau_slug)
                
            elif item.suffix == ".txt":
                nom_fichier = item.stem
                partie_titre = nom_fichier.split(';')[0].strip()
                fiche_slug = clean_slug(partie_titre)
                
                if slug_parent:
                    nom_html_cible = f"en_exo_{slug_parent}_fiche-{fiche_slug}.html"
                else:
                    nom_html_cible = f"en_exo_fiche-{fiche_slug}.html"
                
                generer_html_fiche_exo(item, partie_titre, dest_dir / nom_html_cible)


    def generer_html_fiche_exo(fichier_txt, titre_fiche, chemin_html_sortie):
        lignes_exercices_html = []
        
        # --- CALCUL DYNAMIQUE DU FIL D'ARIANE (BREADCRUMBS) ---
        # On récupère la liste des dossiers parents par rapport à la racine "files/Anglais Exo/"
        relative_path = fichier_txt.relative_to(path_exos)
        dossiers_parents = relative_path.parent.parts  # Exemple : ('Media', 'Journalism')
        
        # Structure de base du fil d'Ariane
        breadcrumbs_links = [
            '<a href="../index.html">Accueil</a>',
            '<a href="./en.html">Anglais</a>',
            '<a href="./en_exo.html">Exercices</a>'
        ]
        
        # On reconstruit les liens cliquables pour chaque niveau de dossier traversé
        slugs_accumules = []
        for dossier in dossiers_parents:
            slug_dossier = clean_slug(dossier)
            slugs_accumules.append(slug_dossier)
            
            # Nom de la page de navigation cible (ex: en_exo_media_journalism.html)
            target_nav_html = f"en_exo_{'_'.join(slugs_accumules)}.html"
            nom_affiche = dossier.replace("-", " ").capitalize()
            
            breadcrumbs_links.append(f'<a href="./{target_nav_html}">{nom_affiche}</a>')
            
        # On ajoute le titre de la fiche courante à la fin (non cliquable)
        breadcrumbs_links.append(f'<span>{titre_fiche}</span>')
        breadcrumbs_html = " > ".join(breadcrumbs_links)
        # ------------------------------------------------------
        
        with open(fichier_txt, "r", encoding="utf-8") as f:
            index_exo = 1
            for ligne in f:
                ligne = ligne.strip()
                if not ligne or ligne.startswith("==="):
                    continue
                
                parts = ligne.split(";")
                if len(parts) < 7:
                    continue
                
                # Extraction stricte des 7 colonnes dans l'ordre exact indiqué
                enonce, reponses_brutes, indications, explications, type_exo, provenance, niveau = [p.strip() for p in parts[:7]]
                
                # Extraction des crochets pour les trous
                trous_extraits = re.findall(r'\[(.*?)\]', reponses_brutes)
                reponses_propres = ",".join(trous_extraits)
                
                # Fabrication de la phrase à trous
                phrase_a_trous = reponses_brutes
                for trou in trous_extraits:
                    variantes = trou.split("/")
                    longueur_max = max(len(v) for v in variantes)
                    taille_input = max(7, longueur_max + 3)
                    
                    if taille_input <= 125:
                        html_input = f'<input type="text" class="input-exo-trou" size="{taille_input}" placeholder="Complétez...">'
                    else:
                        html_input = f'<textarea class="input-exo-long" placeholder="Complétez..."></textarea>'
                    
                    phrase_a_trous = phrase_a_trous.replace(f"[{trou}]", html_input)
                
                # Contenu HTML de l'énoncé
                html_body = f'<p class="exo-enonce-text"><strong>Enoncé : </strong>{enonce}</p>'
                html_body += f'<p class="exo-enonce-text">{phrase_a_trous}</p>'
                
                # Échappements HTML
                reponses_clean_esc = reponses_propres.replace('"', '&quot;')
                reponses_brutes_clean = reponses_brutes.replace("[", "").replace("]", "").replace('"', '&quot;')
                
                html_exo = f"""
                <div class="exo-card" data-type="{type_exo}" data-reponses-clean="{reponses_clean_esc}" data-reponses-brutes="{reponses_brutes_clean}" data-index="{index_exo}">
                    <div class="exo-header">
                        <span class="exo-number">Exercice {index_exo}</span>
                        <div class="exo-badges-right">
                            <span class="badge badge-provenance">📚 {provenance}</span>
                            <span class="badge badge-niveau">{niveau}</span>
                            <span class="badge badge-type-exo">{type_exo}</span>
                        </div>
                    </div>
                    <div class="exo-body">
                        {html_body}
                    </div>
                    <div class="exo-actions">
                        <button class="btn-exo btn-indication" onclick="basculerIndication({index_exo})">💡 Indication</button>
                        <button class="btn-exo btn-valider" onclick="verifierExercice({index_exo})">✔ Valider</button>
                    </div>
                    <div class="exo-feedback-box" id="feedback-{index_exo}"></div>
                    
                    <div class="exo-indication-panel hidden" id="indication-{index_exo}">
                        <strong>Indication :</strong> {indications if indications else "Aucune indication disponible."}
                    </div>
                    <div class="exo-explication-panel hidden" id="explication-{index_exo}">
                        <strong>Explication :</strong><br>{explications if explications else "Pas d'explication supplémentaire."}
                    </div>
                </div>
                """
                lignes_exercices_html.append(html_exo)
                index_exo += 1

        contenu_html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Exercices : {titre_fiche} - Prépa Blois</title>
    <link rel="stylesheet" href="../style.css">
</head>
<body>
    <header class="main-header">
        <h1><a href="../index.html">Prépa Blois</a></h1>
    </header>

    <main class="container">
        <h2 class="page-title">ANGLAIS > Exercices</h2>
        <h4 class="breadcrumbs">
            {breadcrumbs_html}
        </h4>
        <div class="fiche-header-container">
            <div class="fiche-titles">
                <p class="intro-text">Fiche d'exercices</p>
                <h3 class="fiche-title">{titre_fiche}</h3>
            </div>
        </div>
        <div class="exos-container">
            {"".join(lignes_exercices_html)}
        </div>
    </main>
    <footer class="main-footer">
        <p>Mis à jour le --/--/---- à --h--</p>
    </footer>
    <script src="../script_exos.js"></script>
</body>
</html>"""

        with open(chemin_html_sortie, "w", encoding="utf-8") as f_out:
            f_out.write(contenu_html)
        print(f"✅ Page d'exercice régénérée : {chemin_html_sortie.name}")

    explorer_et_generer(path_exos)



def generer_navigation_exercices(base_dir="files/Anglais Exo/", output_dir="pages/"):
    path_exos = Path(base_dir)
    dest_dir = Path(output_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    # Cartographie de l'arborescence des exercices
    arborescence_exos = {}
    themes_principaux_exos = set()
    
    for root, dirs, files in os.walk(path_exos):
        path_root = Path(root)
        relative_parts = path_root.relative_to(path_exos).parts
        
        # Si on est à la racine du dossier d'exos
        if not relative_parts:
            for d in dirs:
                themes_principaux_exos.add(d)
                if (d,) not in arborescence_exos:
                    arborescence_exos[(d,)] = {'sous_themes': set(), 'fiches': set()}
            continue

        current_node = relative_parts
        if current_node not in arborescence_exos:
            arborescence_exos[current_node] = {'sous_themes': set(), 'fiches': set()}

        for d in dirs:
            arborescence_exos[current_node]['sous_themes'].add(d)
            child_node = current_node + (d,)
            if child_node not in arborescence_exos:
                arborescence_exos[child_node] = {'sous_themes': set(), 'fiches': set()}
                
        for f in files:
            if f.endswith('.txt'):
                arborescence_exos[current_node]['fiches'].add(f[:-4])

    # 1. Génération de la page racine des exercices (pages/en_exo.html)
    corps_racine = """
    <p class="intro-text">Ici il y a des exercices ou il faut :<br><br>
            - Conjuguer des verbes<br>
            - Compléter des phrases<br>
            - Faire des traductions<br><br>
            Pas vraiment encore, mais ça viendra</p>\n
    """
    corps_racine += '<section class="subject-section">\n<h3>Thèmes d\'exercices</h3>\n<div class="topic-grid">\n'
    
    for t in sorted(themes_principaux_exos):
        theme_slug = formater_nom_fichier(t)
        lien_cible = f"./en_exo_{theme_slug}.html"
        nom_image = f"en_vocab_{theme_slug}.png" 
        
        corps_racine += f"""
        <a href="{lien_cible}" class="subject-topic">
            <span class="topic-title">{t.replace("-", " ").capitalize()}</span>
            <img src="../images/{nom_image}" alt="img {t}" class="topic-image">
        </a>\n"""
        
    corps_racine += '</div>\n</section>\n'
    
    # Fil d'ariane pointant vers en_exo.html (sans lien vers un arbre global)
    breadcrumbs_racine = '<a href="../index.html">Accueil</a> > <a href="./en.html">Anglais</a> > <a href="./en_exo.html">Exercices</a>'
    
    html_racine = TEMPLATE_CONTENT.format(
        PAGE_TITLE="Exercices",
        MAIN_TITLE="ANGLAIS > Exercices",
        BREADCRUMBS=breadcrumbs_racine,
        CONTENT=corps_racine
    )
    
    with open(dest_dir / "en_exo.html", "w", encoding="utf-8") as f_out:
        f_out.write(html_racine)

    # 2. Génération des pages de navigation intermédiaires et des liens de fiches
    for node, data in arborescence_exos.items():
        theme_slugs = [formater_nom_fichier(s) for s in node]
        nom_base_page = "en_exo_" + "_".join(theme_slugs)
        fichier_html_cible = dest_dir / f"{nom_base_page}.html"
        
        titre_navigation = node[-1].replace("-", " ").capitalize()
        corps_page = ""
        
        # S'il y a des sous-dossiers
        if data['sous_themes']:
            corps_page += '<section class="subject-section">\n<h3>Sous-thèmes disponibles</h3>\n<div class="topic-grid">\n'
            for st in sorted(data['sous_themes']):
                st_slug = formater_nom_fichier(st)
                lien_cible = f"./{nom_base_page}_{st_slug}.html"
                nom_image = f"en_vocab_{st_slug}.png"
                
                corps_page += f"""
                <a href="{lien_cible}" class="subject-topic">
                    <span class="topic-title">{st.replace("-", " ").capitalize()}</span>
                    <img src="../images/{nom_image}" alt="img {st}" class="topic-image">
                </a>\n"""
            corps_page += '</div>\n</section>\n'
            
        # S'il y a des fichiers .txt -> Génération des liens au format strict demandé
        if data['fiches']:
            corps_page += '<section class="fiches-section">\n<h3>Fiches d\'exercices</h3>\n<ul class="subject-list">\n'
            for fiche in sorted(data['fiches']):
                partie_titre = fiche.split(';')[0].strip()
                fiche_slug = formater_nom_fichier(partie_titre)
                
                # Format demandé : en_exo_THEME_SOUS-THEME_fiche-NOM-FICHE.html
                nom_fiche_html = f"{nom_base_page}_fiche-{fiche_slug}.html"
                titre_fiche = partie_titre.replace("-", " ").capitalize()
                
                corps_page += f'<li><a href="./{nom_fiche_html}" class="subject-link">📝 {titre_fiche}</a></li>\n'
            corps_page += '</ul>\n</section>\n'
        
        # Fil d'ariane dynamique (sans référence à en_tree.html)
        breadcrumbs_list = ['<a href="../index.html">Accueil</a>', '<a href="./en.html">Anglais</a>', '<a href="./en_exo.html">Exercices</a>']
        for i in range(len(node)):
            current_node = node[:i+1]
            target_file = "en_exo_" + "_".join([formater_nom_fichier(s) for s in current_node]) + ".html"
            titre_segment = current_node[-1].replace("-", " ").capitalize()
            
            if i == len(node) - 1:
                breadcrumbs_list.append(f'<span>{titre_segment}</span>')
            else:
                breadcrumbs_list.append(f'<a href="./{target_file}">{titre_segment}</a>')
                
        breadcrumbs_html = " > ".join(breadcrumbs_list)
        
        html_final = TEMPLATE_CONTENT.format(
            PAGE_TITLE=f"Exercices - {titre_navigation}",
            MAIN_TITLE="ANGLAIS > Exercices",
            BREADCRUMBS=breadcrumbs_html,
            CONTENT=corps_page
        )
        
        with open(fichier_html_cible, "w", encoding="utf-8") as f_out:
            f_out.write(html_final)
            
    print("✅ Navigation en_exo.html et pages intermédiaires générées au format demandé !")






generer_fiches_exercices()
generer_navigation_exercices()
mettre_a_jour_footers()

✅ Page d'exercice régénérée : en_exo_media_the-press_fiche-exos-journalisme-1.html
✅ Navigation en_exo.html et pages intermédiaires générées au format demandé !

Terminé ! 159 page(s) mise(s) à jour avec le message : Mis à jour le 08/07/2026 à 18h39


In [19]:
"""
  _______  __   __  _______          ____    _    _   _____   ______  _        ______   _______ 
 |__   __| \ \ / / |__   __|        / __ \  | |  | | |_   _| |___  / | |      |  ____| |__   __|
    | |     \ V /     | |          | |  | | | |  | |   | |      / /  | |      | |__       | |   
    | |      > <      | |          | |  | | | |  | |   | |     / /   | |      |  __|      | |   
    | |     / . \     | |          | |__| | | |__| |  _| |_   / /__  | |____  | |____     | |   
    |_|    /_/ \_\    |_|           \___\_\  \____/  |_____| /_____| |______| |______|    |_|   
"""





import os
import re
from pathlib import Path

# --- Configuration des chemins ---
ROOT_DIR = Path.cwd().absolute()
FILES_DIR = ROOT_DIR / "files" / "Anglais"
OUTPUT_DIR = ROOT_DIR / "texts-quizlet-lists"

# Créer le dossier de destination s'il n'existe pas
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def formater_nom_fichier(chaine):
    """Nettoie une chaîne pour en faire un nom de fichier propre (slug)"""
    chaine = chaine.lower()
    chaine = re.sub(r'[^\w\s-]', '', chaine)
    return re.sub(r'[-\s]+', '-', chaine).strip('-')

def construire_nom_liste(node, fiche):
    """Génère le nom du fichier .txt pour Anki sans la partie française"""
    theme_slugs = [formater_nom_fichier(s) for s in node]
    
    # RETRAIT DU FRANÇAIS : On ne garde que l'anglais avant le ';'
    if ';' in fiche:
        fiche = fiche.split(';', 1)[0].strip()
        
    nom_base = "en_vocab_" + "_".join(theme_slugs)
    nom_base += f"_liste-{formater_nom_fichier(fiche)}"
    return f"{nom_base}.txt"

# --- Parcours de l'arborescence et conversion ---
compteur = 0

for root, dirs, files in os.walk(FILES_DIR):
    path_root = Path(root)
    relative_parts = path_root.relative_to(FILES_DIR).parts
    
    # On traite uniquement les fichiers dans les sous-dossiers thématiques
    if not relative_parts:
        continue

    for f in files:
        if f.endswith('.txt'):
            nom_fiche_sans_ext = f[:-4]
            chemin_txt_origine = path_root / f
            
            # Liste pour stocker les blocs de mots formatés
            blocs_mots = []
            
            with open(chemin_txt_origine, "r", encoding="utf-8") as f_in:
                for line in f_in:
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    
                    # Ignorer la ligne de lien si elle existe
                    if line.startswith("QUIZLET;") or line.startswith("LIEN;") or line.startswith("http"):
                        continue
                    
                    # Découpage des 7 colonnes standard
                    parts = [p.strip() for p in line.split(';')]
                    if len(parts) < 7:
                        continue
                    
                    anglais = parts[0]
                    francais = parts[1]
                    note = parts[2]
                    exemple = parts[3]
                    
                    # Remplacement des balises HTML <br> par des retours à la ligne
                    anglais = re.sub(r'<br\s*/?>', '\n', anglais)
                    francais = re.sub(r'<br\s*/?>', '\n', francais)
                    
                    # Traitement de l'exemple (Ajout du préfixe conditionnel)
                    if exemple == "...":
                        exemple_final = ""
                    else:
                        exemple_nettoye = re.sub(r'<br\s*/?>', '\n', exemple)
                        exemple_final = f"\n\nExemple : {exemple_nettoye}"
                        
                    # Traitement de la note (Ajout du préfixe conditionnel)
                    if note == "...":
                        note_final = ""
                    else:
                        note_nettoye = re.sub(r'<br\s*/?>', '\n', note)
                        note_final = f"\n\nNote : {note_nettoye}"
                    
                    # Construction du format individuel :
                    # {anglais}\n\nExemple : {exemple}||{français}\n\nNote : {note}
                    partie_gauche = f"{anglais}{exemple_final}".strip()
                    partie_droite = f"{francais}{note_final}".strip()
                    
                    bloc_formate = f"{partie_gauche}||{partie_droite}"
                    blocs_mots.append(bloc_formate)
            
            # Si le fichier contenait des mots valides, on l'exporte
            if blocs_mots:
                # Jointure de tous les blocs par "\n----------\n"
                contenu_final = "\n----------\n".join(blocs_mots)
                
                # Détermination du nom approprié basé sur l'arborescence
                nom_fichier_cible = construire_nom_liste(relative_parts, nom_fiche_sans_ext)
                chemin_cible = OUTPUT_DIR / nom_fichier_cible
                
                with open(chemin_cible, "w", encoding="utf-8") as f_out:
                    f_out.write(contenu_final)
                
                print(f"Créé : {nom_fichier_cible}")
                compteur += 1

print(f"\nTerminé ! {compteur} fichiers ont été convertis et sauvegardés dans /texts-lists")

Créé : en_vocab_education_assessment-and-achievement_liste-abilities.txt
Créé : en_vocab_education_assessment-and-achievement_liste-achievement.txt
Créé : en_vocab_education_assessment-and-achievement_liste-assessment.txt
Créé : en_vocab_education_assessment-and-achievement_liste-knowledge.txt
Créé : en_vocab_education_learning-difficulties_liste-academic-failure.txt
Créé : en_vocab_education_learning-difficulties_liste-learning-disabilities.txt
Créé : en_vocab_education_the-educational-system_liste-higher-education.txt
Créé : en_vocab_education_the-educational-system_liste-preschool-primary-and-secondary.txt
Créé : en_vocab_education_the-educational-system_liste-school-life.txt
Créé : en_vocab_education_the-educational-system_liste-training.txt
Créé : en_vocab_immigration_liste-references-places-nicknames.txt
Créé : en_vocab_immigration_dreams-of-a-better-life_liste-expectations-and-dreams.txt
Créé : en_vocab_immigration_dreams-of-a-better-life_liste-reasons-to-escape.txt
Créé : en_vo

In [20]:
"""
  _______  __   __  _______                    _   _   _  __  _____ 
 |__   __| \ \ / / |__   __|           /\     | \ | | | |/ / |_   _|
    | |     \ V /     | |             /  \    |  \| | | ' /    | |  
    | |      > <      | |            / /\ \   | . ` | |  <     | |  
    | |     / . \     | |           / ____ \  | |\  | | . \   _| |_ 
    |_|    /_/ \_\    |_|          /_/    \_\ |_| \_| |_|\_\ |_____|
"""





import os
import re
from pathlib import Path

# --- Configuration des chemins ---
ROOT_DIR = Path.cwd().absolute()
FILES_DIR = ROOT_DIR / "files" / "Anglais"
OUTPUT_DIR = ROOT_DIR / "texts-anki-lists"

# Créer le dossier de destination pour Anki s'il n'existe pas
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def formater_nom_fichier(chaine):
    """Nettoie une chaîne pour en faire un nom de fichier propre (slug)"""
    chaine = chaine.lower()
    chaine = re.sub(r'[^\w\s-]', '', chaine)
    return re.sub(r'[-\s]+', '-', chaine).strip('-')

def formater_tag_anki(chaine):
    """Nettoie une chaîne pour créer un tag Anki valide (sans espace ni caractères spéciaux)"""
    # Remplacer les espaces et tirets multiples par un seul tiret
    chaine = re.sub(r'[-\s]+', '-', chaine)
    # Supprimer tout ce qui n'est pas alphanumérique ou un tiret/underscore
    return re.sub(r'[^\w-]', '', chaine)

def construire_nom_liste(node, fiche):
    """Génère le nom du fichier .txt pour Anki sans la partie française"""
    theme_slugs = [formater_nom_fichier(s) for s in node]
    
    # RETRAIT DU FRANÇAIS : On ne garde que l'anglais avant le ';'
    if ';' in fiche:
        fiche = fiche.split(';', 1)[0].strip()
        
    nom_base = "en_vocab_" + "_".join(theme_slugs)
    nom_base += f"_liste-{formater_nom_fichier(fiche)}"
    return f"{nom_base}.txt"

def convertir_type(v_type):
    """Mapping des types pour correspondre aux libellés standard des badges (en minuscules pour Anki)"""
    v_type = v_type.strip().upper()
    mapping = {
        'N': 'noun', 'V': 'verb', 'VI': 'irr. verb',
        'ADJ': 'adj', 'ABREV': 'abrev', 'A': 'other', 'DEF': 'notion'
    }
    return mapping.get(v_type, v_type.lower())

# --- Parcours de l'arborescence et conversion ---
compteur = 0

for root, dirs, files in os.walk(FILES_DIR):
    path_root = Path(root)
    relative_parts = path_root.relative_to(FILES_DIR).parts
    
    # On ignore la racine
    if not relative_parts:
        continue

    for f in files:
        if f.endswith('.txt'):
            nom_fiche_sans_ext = f[:-4]
            chemin_txt_origine = path_root / f
            
            lignes_anki = []
            
            with open(chemin_txt_origine, "r", encoding="utf-8") as f_in:
                for line in f_in:
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    
                    # Ignorer la ligne de lien (Quizlet/Lien/http)
                    if line.startswith("QUIZLET;") or line.startswith("LIEN;") or line.startswith("http"):
                        continue
                    
                    # Découpage des 7 colonnes standard
                    parts = [p.strip() for p in line.split(';')]
                    if len(parts) < 7:
                        continue
                    
                    anglais = parts[0]
                    francais = parts[1]
                    note = parts[2]
                    exemple = parts[3]
                    v_type = parts[4]
                    provenance = parts[5]
                    niveau = parts[6]
                    
                    # --- 1. Construction du RECTO (Field 1 : Anglais + Exemple) ---
                    recto = anglais
                    if exemple != "...":
                        recto += f"<br><br>Exemple :<br>{exemple}"
                    
                    # --- 2. Construction du VERSO (Field 2 : Français + Note) ---
                    verso = francais
                    if note != "...":
                        verso += f"<br><br>Note :<br>{note}"
                    
                    # --- 3. Construction des TAGS (Field 3) ---
                    tags_list = []
                    
                    # Tag de niveau
                    if niveau != "...":
                        tags_list.append(f"ANGLAIS::niveau::{formater_tag_anki(niveau)}")
                        
                    # Tag de type de mot
                    type_propre = convertir_type(v_type)
                    if type_propre != "...":
                        tags_list.append(f"ANGLAIS::type::{formater_tag_anki(type_propre)}")
                    
                    # Tag de l'arborescence thématique
                    # Extraction uniquement de la partie anglaise du nom du fichier si point-virgule
                    nom_liste_propre = nom_fiche_sans_ext.split(';')[0].strip()
                    
                    segments_tags = [formater_tag_anki(s) for s in relative_parts] + [formater_tag_anki(nom_liste_propre)]
                    tag_arborescence = "ANGLAIS::" + "::".join(segments_tags)
                    tags_list.append(tag_arborescence)
                    
                    # Dans Anki, les différents tags attachés à une carte sont séparés par un espace
                    string_tags = " ".join(tags_list)
                    
                    # Assemblage de la ligne Anki : Recto;Verso;Tags
                    lignes_anki.append(f"{recto};{verso};{string_tags}")
            
            # Si la liste contient des cartes valides, on l'exporte
            if lignes_anki:
                # Dans un fichier d'import Anki standard, chaque carte est séparée par un saut de ligne classique
                contenu_final = "\n".join(lignes_anki)
                
                # Détermination du nom approprié basé sur l'arborescence
                nom_fichier_cible = construire_nom_liste(relative_parts, nom_fiche_sans_ext)
                chemin_cible = OUTPUT_DIR / nom_fichier_cible
                
                with open(chemin_cible, "w", encoding="utf-8") as f_out:
                    f_out.write(contenu_final)
                
                print(f"Créé pour Anki : {nom_fichier_cible}")
                compteur += 1

print(f"\nTerminé ! {compteur} fichiers prêts pour Anki ont été sauvegardés dans /texts-anki-lists")

Créé pour Anki : en_vocab_education_assessment-and-achievement_liste-abilities.txt
Créé pour Anki : en_vocab_education_assessment-and-achievement_liste-achievement.txt
Créé pour Anki : en_vocab_education_assessment-and-achievement_liste-assessment.txt
Créé pour Anki : en_vocab_education_assessment-and-achievement_liste-knowledge.txt
Créé pour Anki : en_vocab_education_learning-difficulties_liste-academic-failure.txt
Créé pour Anki : en_vocab_education_learning-difficulties_liste-learning-disabilities.txt
Créé pour Anki : en_vocab_education_the-educational-system_liste-higher-education.txt
Créé pour Anki : en_vocab_education_the-educational-system_liste-preschool-primary-and-secondary.txt
Créé pour Anki : en_vocab_education_the-educational-system_liste-school-life.txt
Créé pour Anki : en_vocab_education_the-educational-system_liste-training.txt
Créé pour Anki : en_vocab_immigration_liste-references-places-nicknames.txt
Créé pour Anki : en_vocab_immigration_dreams-of-a-better-life_liste-